### Main ###

In [ ]:
from pathlib import Path
import numpy as np

from pySPADutils import pySPADutils as util
from bootstrapping import groupBP

In [ ]:
def process_spad_folders(
    paths,
    files=None,
    
    flag_saveBP=0,
    flag_preview=0,
    
    flag_ROIonly=0,
    rois=None,

    flag_grouping=0,
    groups=None,
):

    for folder in paths:
        folder = Path(folder)
        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue

        if files is None:
            file_list = sorted(folder.glob("*.bin"))
        else:
            file_list = [folder / f for f in files]

        for file_path in file_list:
            if not file_path.exists():
                print(f"Skipping missing file: {file_path}")
                continue

            print(f"Processing: {file_path}")
            
            for f in files:
                print(f"Reading {f}")
                bytearrayFile = util.readBinBig(file_path)

                print("Unpacking the bin file")
                img_full = util.unpackBytearray(bytearrayFile)  # shape: (frames, Y, X)

                # Match sCMOS orientation
                img_full = np.flip(img_full, axis=1)
                img_full = np.rot90(img_full, k=1, axes=(1, 2))

                # If no ROI mode, process the full image as one ROI
                if flag_ROIonly:
                    roi_list = rois
                else:
                    roi_list = [("full", 0, 0, img_full.shape[2], img_full.shape[1])]
                

                for roi_name, ystart, xstart, width, height in roi_list:
                    print(f"Processing {roi_name} with coordinates (x={xstart}, y={ystart}, width={width}, height={height})")

                    img = img_full[:, xstart - 1:xstart + width - 1, ystart - 1:ystart + height - 1]   # -1 for 1-based to 0-based indexing

                    if flag_saveBP:
                        out_path = file_path.with_name(file_path.stem + f"_{roi_name}_BP.tif")
                        util.writeTiffBig(
                            out_path,
                            img,
                            dtype=np.uint8
                        )

                    if flag_preview:
                        z_sum = np.sum(img, axis=0, dtype=np.uint16)
                        out_path = file_path.with_name(file_path.stem + f"_{roi_name}_ZSum.tif")
                        util.writeTiffBig(
                            out_path,
                            z_sum
                        )

                    if flag_grouping:
                        for mode, total_BP, out_frames, bp_per_frame in groups:
                            print(
                                f"Grouping {roi_name}: mode={mode}, "
                                f"out_frames={out_frames}, "
                                f"bp_per_frame={bp_per_frame}"
                                f" (total_BP={total_BP}) "
                            )

                            img_grouped, idx = groupBP(
                                img[:total_BP],
                                mode=mode,
                                out_frames=out_frames,
                                bp_per_frame=int(bp_per_frame)
                            )

                            out_path = file_path.with_name(file_path.stem + f"_{roi_name}_gpmode{mode}_{out_frames}fr_{bp_per_frame}bppfr_ttl{total_BP}bp.tif")
                            util.writeTiffBig(out_path, img_grouped)

                print(f"Done!\n")

if __name__ == "__main__":

    paths1 = [
        "H:\\_data\\KH260421_2D_beads_and_rings\\rings1um-ex488\\SPAD\\FOV_1",
    ]
    rois1 = [
        ("ROI0101", 273, 95, 60, 60),
        ("ROI0102", 306, 367, 60, 60),
        ("ROI0103", 382, 356, 60, 60),
    ]


    paths2 = [
        "H:\\_data\\KH260421_2D_beads_and_rings\\rings1um-ex488\\SPAD\\FOV_2",
    ]
    rois2 = [
        ("ROI0201", 241, 237, 60, 60),
        ("ROI0202", 244, 348, 60, 60),
        ("ROI0203", 189, 111, 60, 60),
    ]
	

    paths3 = [
        "H:\\_data\\KH260421_2D_beads_and_rings\\rings1um-ex488\\SPAD\\FOV_3",
    ]
    rois3 = [
        ("ROI0301", 200, 199, 60, 60),
        ("ROI0302", 236, 296, 60, 60),
        ("ROI0303", 179, 85, 60, 60),
    ]
	
	
    paths4 = [
        "H:\\_data\\KH260421_2D_beads_and_rings\\rings1um-ex488\\SPAD\\FOV_4",
    ]
    rois4 = [
        ("ROI0401", 87,  253, 60, 60),
        ("ROI0402", 346, 191, 60, 60),
        ("ROI0403", 328, 371, 60, 60),
    ]

    paths5 = [
        "H:\\_data\\KH260421_2D_beads_and_rings\\rings1um-ex488\\SPAD\\FOV_5",
    ]
    rois5 = [
        ("ROI0501", 123, 297, 60, 60),
        ("ROI0502", 198, 131, 60, 60),
        ("ROI0503", 266, 193, 60, 60),
    ]


    files = [f"Z_{i}.bin" for i in range(50000, 50001, 100)]


    flag_ROIonly = 1        # Define ROIs as (name, ystart, xstart, width, height)

    flag_saveBP = 0
    flag_preview = 1
    flag_grouping = 1       # define groupings as (mode, total_BP, out_frames, bp_per_frame)
    

    # TODO: change total BP to the last item
    groups = [
        (1, 100000, 20, 5000),

        (4, 100000, 2, 100000),
        (4, 100000, 5, 100000),
        (4, 100000, 10, 100000),
        (4, 100000, 20, 100000),
        (4, 100000, 50, 100000),
        (4, 100000, 100, 100000),
        (4, 100000, 200, 100000),
        (4, 100000, 500, 100000),
        (4, 100000, 1000, 100000),

        (4, 100000, 20, 100),
        (4, 100000, 20, 200),
        (4, 100000, 20, 500),
        (4, 100000, 20, 1000),
        (4, 100000, 20, 2000),
        (4, 100000, 20, 5000),
        (4, 100000, 20, 10000),
        (4, 100000, 20, 20000),
        (4, 100000, 20, 50000),
        (4, 100000, 20, 100000),
        (4, 100000, 20, 200000),

        (4, 100, 20, 100000),
        (4, 200, 20, 100000),
        (4, 500, 20, 100000),
        (4, 1000, 20, 100000),
        (4, 2000, 20, 100000),
        (4, 5000, 20, 100000),
        (4, 10000, 20, 100000),
        (4, 20000, 20, 100000),
        (4, 50000, 20, 100000),
        (4, 100000, 20, 100000),

    ]

    # Varying total BP
    groups2 = [
        (4, 100, 20, 100000),
        (4, 200, 20, 100000),
        (4, 500, 20, 100000),
        (4, 1000, 20, 100000),
        (4, 2000, 20, 100000),
        (4, 5000, 20, 100000),
        (4, 10000, 20, 100000),
        (4, 20000, 20, 100000),
        (4, 50000, 20, 100000),
        (4, 100000, 20, 100000),

        ]
    
    groups3 = [
        (4, 100, 100, 100000),
        (4, 200, 100, 100000),
        (4, 500, 100, 100000),
        (4, 1000, 100, 100000),
        (4, 2000, 100, 100000),
        (4, 5000, 100, 100000),
        (4, 10000, 100, 100000),
        (4, 20000, 100, 100000),
        (4, 50000, 100, 100000),
        (4, 100000, 100, 100000),
    ]


#    process_spad_folders(paths1, files=files, rois=rois1, groups=groups, flag_ROIonly=flag_ROIonly, flag_saveBP=flag_saveBP, flag_preview=flag_preview, flag_grouping=flag_grouping)
#    process_spad_folders(paths2, files=files, rois=rois2, groups=groups, flag_ROIonly=flag_ROIonly, flag_saveBP=flag_saveBP, flag_preview=flag_preview, flag_grouping=flag_grouping)
#    process_spad_folders(paths3, files=files, rois=rois3, groups=groups, flag_ROIonly=flag_ROIonly, flag_saveBP=flag_saveBP, flag_preview=flag_preview, flag_grouping=flag_grouping)
    process_spad_folders(paths4, files=files, rois=rois4, groups=groups3, flag_ROIonly=flag_ROIonly, flag_saveBP=flag_saveBP, flag_preview=flag_preview, flag_grouping=flag_grouping)
#    process_spad_folders(paths5, files=files, rois=rois5, groups=groups, flag_ROIonly=flag_ROIonly, flag_saveBP=flag_saveBP, flag_preview=flag_preview, flag_grouping=flag_grouping)